# Auralis stage 2: continue training on train-clean-100 + train-other-500 (Kaggle)

Continues from the stage 1 checkpoint (`sagexd08/auralis-train-ls460`, attached as a kernel input) with a fresh optimizer and a lower peak learning rate. The mix is about 600 hours and about 36 GB, which fits Kaggle's scratch disk. Together with stage 1 the model sees all 960 LibriSpeech hours.

Run it with **Save Version > Save & Run All (Commit)** or push it with `kaggle kernels push -p training/notebooks/stage2`. Training stops itself after `TRAIN_SECONDS` with the learning rate fully decayed, then scores test-clean and test-other and exports `auralis.onnx`.


In [ ]:
import json, os, shutil, subprocess

SUBSETS = ['train-clean-100', 'train-other-500']
TEST_SETS = ['test-clean', 'test-other']
MAX_STEPS = 80000
TRAIN_SECONDS = 30600
RUN_NAME = 'ls-stage2-001'

TEMP = '/kaggle/temp'
LS = f'{TEMP}/librispeech'
WORK = f'{TEMP}/work'
OUT_DIR = f'/kaggle/working/runs/{RUN_NAME}'
for d in (LS, WORK, OUT_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
!cd /kaggle/temp && rm -rf Auralis && git clone --depth 1 --branch feat/train-recipe https://github.com/Sagexd08/Auralis.git
!nvidia-smi -L
!pip install -q torch --index-url https://download.pytorch.org/whl/cu126
!pip install -q soundfile scipy onnx onnxscript onnxruntime jiwer
import sys
if 'torch' in sys.modules:
    raise RuntimeError('torch was already imported in this session. Run > Restart session, then run all cells again.')
import torch
print(torch.__version__, 'cuda build:', torch.version.cuda)
assert torch.cuda.is_available(), 'torch cannot see a GPU. If nvidia-smi -L above printed no GPU, set Accelerator to GPU T4 x2 in the notebook settings.'
print(torch.cuda.get_device_name(0))


In [ ]:
import glob

def newest(paths):
    paths = [p for p in paths if os.path.isdir(p)]
    return max(paths, key=lambda p: int(p.rsplit('-', 1)[1])) if paths else None

INIT_CKPT = newest(glob.glob('/kaggle/input/**/auralis-stt-step-*', recursive=True))
assert INIT_CKPT, 'stage 1 checkpoint not found: attach sagexd08/auralis-train-ls460 as an input'
print('warm start from', INIT_CKPT)


In [ ]:
import hashlib
import tarfile
import urllib.request

REGISTRY = f'{WORK}/registry.jsonl'
DATA_CLI = ['python', '-m', 'auralis_data', '--registry', REGISTRY]
DATA_DIR = '/kaggle/temp/Auralis/data'
CHECKSUMS = {}

def run(cmd, cwd):
    subprocess.run(cmd, cwd=cwd, check=True)

class HashingReader:
    def __init__(self, raw):
        self.raw = raw
        self.digest = hashlib.sha256()

    def read(self, size=-1):
        block = self.raw.read(size)
        self.digest.update(block)
        return block

def stream_extract(url, dest):
    with urllib.request.urlopen(url) as response:
        reader = HashingReader(response)
        with tarfile.open(fileobj=reader, mode='r|gz') as tar:
            tar.extractall(dest, filter='data')
        while reader.read(1 << 20):
            pass
        return reader.digest.hexdigest()

def pull_and_import(name):
    root = f'{LS}/LibriSpeech/{name}'
    if not os.path.isdir(root):
        CHECKSUMS[name] = stream_extract(f'https://www.openslr.org/resources/12/{name}.tar.gz', LS)
        print('pulled', name, 'sha256', CHECKSUMS[name])
    run(DATA_CLI + ['import', 'librispeech', root, '--id', f'librispeech-{name}', '--language', 'en', '--revision', name, '--manifests', f'{WORK}/manifests'], DATA_DIR)

for name in SUBSETS + TEST_SETS:
    pull_and_import(name)
    print('ready', name)

os.makedirs('/kaggle/working', exist_ok=True)
with open('/kaggle/working/checksums.json', 'w') as f:
    json.dump(CHECKSUMS, f, indent=2)


In [ ]:
def build(corpus_name, datasets, max_duration):
    cfg = {'name': corpus_name, 'datasets': datasets, 'commercial': True, 'min_duration': 1.0, 'max_duration': max_duration, 'min_quality': 0.8, 'seed': 1004}
    path = f'{WORK}/{corpus_name}.config.json'
    with open(path, 'w') as f:
        json.dump(cfg, f)
    run(DATA_CLI + ['build', path, '--manifests', f'{WORK}/manifests', '--out', f'{WORK}/corpora'], DATA_DIR)

build('librispeech-train', [f'librispeech-{n}' for n in SUBSETS], 16.0)
for t in TEST_SETS:
    build(f'librispeech-{t}', [f'librispeech-{t}'], 35.0)

In [ ]:
train_cfg = {
    'corpus_dir': f'{WORK}/corpora',
    'corpus_name': 'librispeech-train',
    'out_dir': OUT_DIR,
    'vocab_size': 256,
    'batch_size': 32,
    'grad_accum': 2,
    'lr': 0.0004,
    'warmup_steps': 500,
    'max_steps': MAX_STEPS,
    'eval_every': 1000,
    'val_pct': 1,
    'max_seconds': 16.0,
    'num_workers': 4,
    'amp': True,
    'spec_augment': True,
    'init_from': INIT_CKPT,
    'time_budget': TRAIN_SECONDS,
    'model': {'d_model': 320, 'n_layers': 14, 'n_heads': 4, 'conv_kernel': 15, 'dropout': 0.1},
}
with open(f'{WORK}/train.json', 'w') as f:
    json.dump(train_cfg, f)

In [ ]:
!cd /kaggle/temp/Auralis/training && timeout {TRAIN_SECONDS + 900} python -u -m auralis_stt.train {WORK}/train.json --resume ; echo "training process ended with status $?"

In [ ]:
import sys
sys.path.insert(0, '/kaggle/temp/Auralis/training')
from torch.utils.data import DataLoader
from auralis_stt.checkpoint import latest_checkpoint, load_model
from auralis_stt.dataset import SpeechDataset, collate, load_corpus
from auralis_stt.features import LogMel
from auralis_stt.train import evaluate

ckpt = latest_checkpoint(OUT_DIR)
print('latest checkpoint', ckpt)
for old in sorted(glob.glob(f'{OUT_DIR}/auralis-stt-step-*'), key=lambda p: int(p.rsplit('-', 1)[1]))[:-2]:
    shutil.rmtree(old)

model, tokenizer = load_model(ckpt, 'cuda')
logmel = LogMel(n_mels=model.cfg.n_mels).to('cuda')
results = {}
for t in TEST_SETS:
    meta, rows, roots = load_corpus(f'{WORK}/corpora', f'librispeech-{t}')
    loader = DataLoader(SpeechDataset(rows, roots, tokenizer, 35.0), 16, shuffle=False, collate_fn=collate, num_workers=2)
    metrics, _, _ = evaluate(model, tokenizer, loader, logmel, torch.device('cuda'))
    results[t] = metrics
    print(t, metrics)
with open(f'{OUT_DIR}/test_results.json', 'w') as f:
    json.dump({'checkpoint': str(ckpt), 'decoding': 'greedy CTC', 'results': results}, f, indent=2)

In [ ]:
from auralis_stt.export import export_checkpoint
EXPORT_DIR = f'/kaggle/working/export/{RUN_NAME}'
onnx_path = export_checkpoint(ckpt, EXPORT_DIR, 'auralis', ['en'])
shutil.copy(f'{OUT_DIR}/test_results.json', EXPORT_DIR)
shutil.make_archive(f'/kaggle/working/auralis-{RUN_NAME}', 'zip', EXPORT_DIR)
print(onnx_path)

## Getting the model back

After the run finishes, open the notebook's **Output** tab and download `auralis-ls-460h-001.zip`. It holds `auralis.onnx`, `auralis.tokenizer.json`, `auralis.json` and `test_results.json`. Put them together in `models/` in the repo and score with `auralis-bench-cli --model models/auralis.onnx --input <wav>`.